In [ ]:
import pandas as pd
import numpy as np

In [ ]:
gold_df = pd.read_csv('../data/3yrs_10k_gold_paths.csv')
silver_df = pd.read_csv('../data/3yrs_10k_silver_paths.csv')

In [ ]:
gold_df

In [ ]:
per_rupee_gold = 1 / gold_df
per_rupee_silver = 1 / silver_df

In [ ]:
per_rupee_gold

In [ ]:
per_rupee_silver

In [ ]:
gold_units_df = pd.DataFrame(np.mean(per_rupee_gold, axis=1), columns=['mean'])
gold_units_df['median'] = np.median(per_rupee_gold, axis=1)
gold_units_df['5perc'] = np.percentile(per_rupee_gold, 5, axis=1)
gold_units_df['95perc'] = np.percentile(per_rupee_gold, 95, axis=1)

display(gold_units_df)


In [ ]:
silver_units_df = pd.DataFrame(np.mean(per_rupee_silver, axis=1), columns=['mean'])
silver_units_df['median'] = np.median(per_rupee_silver, axis=1)
silver_units_df['5perc'] = np.percentile(per_rupee_silver, 5, axis=1)
silver_units_df['95perc'] = np.percentile(per_rupee_silver, 95, axis=1)

display(silver_units_df)


In [ ]:
np.sum(gold_units_df['median'] * (100 - 1))

In [ ]:
gold_units_df['5perc'].iloc[0]

In [ ]:
def invest(total_monthly_amount, gold_ratio=0.5, period='daily', txn_cost=1):

    # Default monthly
    investment_idx = [i for i in range(np.random.randint(1, 30), len(gold_units_df), 30)] #Days of investment
    inv_amt = total_monthly_amount - txn_cost

    if period == 'daily':
        investment_idx = [i for i in range(0, len(gold_units_df), 1)] # Days of investment
        inv_amt = (total_monthly_amount / 30) - txn_cost

    if period == 'weekly':
        investment_idx = [i for i in range(np.random.randint(1, 7), len(gold_units_df), 7)]
        inv_amt = (total_monthly_amount / 7) - txn_cost

    gold_inv_amt = inv_amt * gold_ratio
    silver_inv_amt = inv_amt - gold_inv_amt


    # buy units
    avg_gold_units_acc = 0
    avg_silver_units_acc = 0

    at_risk_gold_units_acc = 0
    at_risk_silver_units_acc = 0

    for i in investment_idx:
        avg_gold_units_acc = avg_gold_units_acc + (gold_units_df['median'].iloc[i] * gold_inv_amt)
        avg_silver_units_acc = avg_silver_units_acc + (silver_units_df['median'].iloc[i] * silver_inv_amt)

        at_risk_gold_units_acc = at_risk_gold_units_acc + (gold_units_df['5perc'].iloc[i] * gold_inv_amt)
        at_risk_silver_units_acc = at_risk_silver_units_acc + (silver_units_df['5perc'].iloc[i] * silver_inv_amt)

    
    ret_dict = {
        'total_monthly_amount': total_monthly_amount,
        'period': period,
        'txn_cost': txn_cost,
        
        'inv_amt': inv_amt,
        'gold_ratio': gold_ratio,
        'silver_ratio': 1 - gold_ratio,

        'avg_gold_units': avg_gold_units_acc,
        'avg_silver_units': avg_silver_units_acc,

        'at_risk_gold_units': at_risk_gold_units_acc,
        'at_risk_silver_units': at_risk_silver_units_acc
    }

    return ret_dict
    

In [ ]:
50 * 30 * 7

In [ ]:
10500 / 7

In [ ]:
10500 / 30

In [ ]:
invest(10500, 0.5, period='weekly')

In [ ]:
%%time
monthly_inv_amt = 10500
txn_cost = 1
period_options = ['daily', 'weekly', 'monthly']
gold_ratio_option = np.arange(0, 1, 0.1)

inv_df = []

for p in period_options:
    for gr in gold_ratio_option:
        inv_df.append(invest(monthly_inv_amt, gold_ratio=gr, period=p, txn_cost=txn_cost))

In [ ]:
inv_df = pd.DataFrame(inv_df)

In [ ]:
inv_df

In [ ]:
avg_gold_price_eoip = gold_df.iloc[-1].median()
at_risk_gold_price_eoip = np.percentile(gold_df.iloc[-1], 5)

avg_silver_price_eoip = silver_df.iloc[-1].median()
at_risk_silver_price_eoip = np.percentile(silver_df.iloc[-1], 5)

In [ ]:
avg_gold_price_eoip

In [ ]:
at_risk_gold_price_eoip

In [ ]:
avg_silver_price_eoip

In [ ]:
at_risk_silver_price_eoip

In [ ]:
inv_df['Avg_Portfolio_Val'] = inv_df['avg_gold_units'] * avg_gold_price_eoip + inv_df['avg_silver_units'] * avg_silver_price_eoip
inv_df['AtRisk_Portfolio_Val'] = inv_df['at_risk_gold_units'] * at_risk_gold_price_eoip + inv_df['at_risk_silver_units'] * at_risk_silver_price_eoip

In [ ]:
inv_df['Risk_to_Reward_Ratio'] = inv_df['AtRisk_Portfolio_Val'] / inv_df['Avg_Portfolio_Val']

In [ ]:
inv_df

In [ ]:
inv_df.iloc[inv_df['Avg_Portfolio_Val'].argmax()]

In [ ]:
inv_df.iloc[inv_df['AtRisk_Portfolio_Val'].argmin()]

In [ ]:
inv_df.iloc[inv_df['Risk_to_Reward_Ratio'].argmin()]